### ORM(Object Relational Mapping)
- 데이터베이스 테이블을 파이썬의 클래스로 매핑
- 컬럼 == 속성
- SQLAIchemy 라이브러리가 ORM 지원

In [1]:
%pip install sqlalchemy


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Users 테이블 (id, name, email)
# create table users()

from sqlalchemy import Column, Integer, String, create_engine
from sqlalchemy.orm import declarative_base, sessionmaker
import os

# 1.x 코드
# 데이터베이스 연결
os.makedirs('db', exist_ok=True)
engine = create_engine('sqlite:///db/user.db', echo=True)

# 모든 모델 클래스의 부모 클래스가 될 Base 객체 생성
Base = declarative_base()

# 모델 클래스: 데이터베이스랑 관련 있는 클래스 
# table로 쓰이면서 실제로 데이터베이스에서 하나의 행을 의미
class User(Base):
  # 테이블 이름 지정
  __tablename__ = "users"
  
  id = Column(Integer, primary_key=True)
  name = Column(String)
  email = Column(String, unique=True)
  
  def __str__(self):
    return f"<User(name='{self.name}', email='{self.email}')>"
  
# 테이블 생성 (없으면 만들고 있으면 만들지 않기)
Base.metadata.create_all(engine)


2026-06-17 09:48:06,804 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-17 09:48:06,806 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("users")
2026-06-17 09:48:06,807 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-06-17 09:48:06,809 INFO sqlalchemy.engine.Engine COMMIT


#### text() 원시 SQL 실행

In [ ]:
# sqlalchemy: DB ORM으로 접근 가능
from sqlalchemy import text

# postgresQL: postgresql:/// 
engine = create_engine('sqlite:///db/demo.db', echo=True)
with engine.connect() as conn:
  conn.execute(text('''
                    CREATE TABLE IF NOT EXISTS users(
                      id INTEGER PRIMARY KEY AUTOINCREMENT, 
                      name TEXT NOT NULL, 
                      age INTEGER
                    )
                    '''))
  conn.commit()
  # 삽입
  conn.execute(
                text('INSERT INTO users(name, age) VALUES (:name, :age)'), 
                [{"name":"Alice", "age":30},{"name":"Bob", "age":25}, ]
              )
  conn.commit()
  # 조회
  result = conn.execute(text("SELECT * FROM users"))
  for row in result:
    print(row.id, row.name, row.age)

2026-06-17 10:01:11,210 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-17 10:01:11,211 INFO sqlalchemy.engine.Engine 
                    CREATE TABLE IF NOT EXISTS users(
                      id INTEGER PRIMARY KEY AUTOINCREMENT, 
                      name TEXT NOT NULL, 
                      age INTEGER
                    )
                    
2026-06-17 10:01:11,211 INFO sqlalchemy.engine.Engine [generated in 0.00109s] ()
2026-06-17 10:01:11,212 INFO sqlalchemy.engine.Engine COMMIT
2026-06-17 10:01:11,214 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-17 10:01:11,215 INFO sqlalchemy.engine.Engine INSERT INTO users(name, age) VALUES (?, ?)
2026-06-17 10:01:11,216 INFO sqlalchemy.engine.Engine [generated in 0.00135s] [('Alice', 30), ('Bob', 25)]
2026-06-17 10:01:11,217 INFO sqlalchemy.engine.Engine COMMIT
2026-06-17 10:01:11,218 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-17 10:01:11,219 INFO sqlalchemy.engine.Engine SELECT * FROM users
2026-06-17 10:01

In [15]:
# engine 에 대한 모든 내용 삭제
Base.metadata.drop_all(engine)

2026-06-17 10:35:33,385 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-17 10:35:33,400 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("users")


2026-06-17 10:35:33,401 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-06-17 10:35:33,417 INFO sqlalchemy.engine.Engine 
DROP TABLE users
2026-06-17 10:35:33,480 INFO sqlalchemy.engine.Engine [no key 0.02237s] ()
2026-06-17 10:35:33,489 INFO sqlalchemy.engine.Engine COMMIT


In [16]:
# 2.x 
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column
from sqlalchemy import DateTime, func
from datetime import datetime

engine = create_engine('sqlite:///db/users.db', echo=True)


class Base(DeclarativeBase):
  pass


class User(Base):
  # 테이블 이름 지정
  __tablename__ = "users"
  
  # 컬러 타입 지정: 타입힌트(파이썬 데이터 타입) 사용
  id:Mapped[int] = mapped_column(primary_key=True, autoincrement=True) # 타입 생략 가능
  name:Mapped[str] # mapped_column 안써도 됨
  email:Mapped[str] = mapped_column(String, unique=True)
  age:Mapped[int]
  created_at:Mapped[datetime]= mapped_column(DateTime, server_default=func.now())
  
  def __str__(self):
    return f"<User(name='{self.name}', email='{self.email}')>"
  
# create_all(): IF NOT EXISTS
Base.metadata.create_all(engine)


2026-06-17 10:36:13,639 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-17 10:36:13,640 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("users")
2026-06-17 10:36:13,641 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-06-17 10:36:13,641 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("users")
2026-06-17 10:36:13,642 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-06-17 10:36:13,643 INFO sqlalchemy.engine.Engine 
CREATE TABLE users (
	id INTEGER NOT NULL, 
	name VARCHAR NOT NULL, 
	email VARCHAR NOT NULL, 
	age INTEGER NOT NULL, 
	created_at DATETIME DEFAULT CURRENT_TIMESTAMP NOT NULL, 
	PRIMARY KEY (id), 
	UNIQUE (email)
)


2026-06-17 10:36:13,644 INFO sqlalchemy.engine.Engine [no key 0.00041s] ()
2026-06-17 10:36:13,649 INFO sqlalchemy.engine.Engine COMMIT


#### session
연결을 계속 연결하고 있는 상태
- 데이터베이스 연동
- 변경사항 추적하고 트랜잭션 관리
- sessionmaker 팩토리 사용

In [18]:
# 세션 팩토리 생성
Session = sessionmaker(bind=engine, autoflush=False, autocommit=False) # 자동 커밋 안함
# 세셕 객체 생성
session = Session() # 세션 객체 생성

In [45]:
session.close()

In [25]:
# insert
# session.add() / session.add_all([]) / session.commit()

# 사용자 생성
new_user = User(name="Alice", email="alice2@example.com", age=35)
session.add(new_user)

In [26]:
session.commit()


2026-06-17 10:46:59,731 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-17 10:46:59,735 INFO sqlalchemy.engine.Engine INSERT INTO users (name, email, age) VALUES (?, ?, ?) RETURNING id, created_at
2026-06-17 10:46:59,736 INFO sqlalchemy.engine.Engine [cached since 432.9s ago] ('Alice', 'alice2@example.com', 35)
2026-06-17 10:46:59,739 INFO sqlalchemy.engine.Engine COMMIT


In [20]:
with Session() as session:
  session.add_all(
    [
      User(name="Bob", email="bob@example.com", age=28),
      User(name="Charlie", email="charlie@example.com", age=35)
    ]
  )
  session.commit()

2026-06-17 10:38:55,309 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-17 10:38:55,317 INFO sqlalchemy.engine.Engine INSERT INTO users (name, email, age) VALUES (?, ?, ?) RETURNING id, created_at
2026-06-17 10:38:55,317 INFO sqlalchemy.engine.Engine [generated in 0.00025s (insertmanyvalues) 1/2 (ordered; batch not supported)] ('Bob', 'bob@example.com', 28)
2026-06-17 10:38:55,320 INFO sqlalchemy.engine.Engine INSERT INTO users (name, email, age) VALUES (?, ?, ?) RETURNING id, created_at
2026-06-17 10:38:55,320 INFO sqlalchemy.engine.Engine [insertmanyvalues 2/2 (ordered; batch not supported)] ('Charlie', 'charlie@example.com', 35)
2026-06-17 10:38:55,330 INFO sqlalchemy.engine.Engine COMMIT


In [36]:
from sqlalchemy import select

with Session() as session:
  
  stmt = select(User) 
  
  # id를 사용해 조회
  # user = session.get(User, 1)
  # print("---단일사용자---")
  # print(user)
  
  # print("---모든사용자---")
  # all_users = session.query(User).all()
  # for user in all_users:
  #   print(user)
  
  # print("---모든사용자 sqlalchemy select---")
  # all_users = session.scalars(stmt).all()
  # for user in all_users:
  #   print(user)
    
  # print("---where---")
  # alice = session.query(User).filter_by(name="Alice").first()
  # print(f"찾은사용자 {alice}")
  
  # print("---where 조건---")
  # select(User).where(User.age >=30)
  # fined_users = session.scalars(stmt).all()
  # for user in fined_users:
  #   print(user)
  
  print("---where like---")
  select(User).where(User.email.like('%ali%'))
  fined_users = session.scalars(stmt).all()
  for user in fined_users:
    print(user)
  
  print("---order by---")
  select(User).order_by(User.id.desc()).limit(2)
  fined_users = session.scalars(stmt).all()
  for user in fined_users:
    print(user)
    
  print("--- 집계함수---")
  sel = select(func.count()).select_from(User)
  count = session.scalars(sel)
  print(count)

---where like---
2026-06-17 10:57:36,162 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-17 10:57:36,164 INFO sqlalchemy.engine.Engine SELECT users.id, users.name, users.email, users.age, users.created_at 
FROM users
2026-06-17 10:57:36,165 INFO sqlalchemy.engine.Engine [cached since 775.1s ago] ()
<User(name='Bob', email='bob@example.com')>
<User(name='Charlie', email='charlie@example.com')>
<User(name='Alice', email='alice.new@example.com')>
<User(name='Alice', email='alice2@example.com')>
---order by---
2026-06-17 10:57:36,167 INFO sqlalchemy.engine.Engine SELECT users.id, users.name, users.email, users.age, users.created_at 
FROM users
2026-06-17 10:57:36,167 INFO sqlalchemy.engine.Engine [cached since 775.1s ago] ()
<User(name='Bob', email='bob@example.com')>
<User(name='Charlie', email='charlie@example.com')>
<User(name='Alice', email='alice.new@example.com')>
<User(name='Alice', email='alice2@example.com')>
--- 집계함수---
2026-06-17 10:57:36,168 INFO sqlalchemy.engine.Engine

In [ ]:
with Session() as session:
  alice = session.query(User).filter_by(name="Alice").first()
  
  alice.email = 'alice.new@example.com'
  session.commit()
  
  print(f"수정된 사용자 {session.query(User).filter_by(name="Alice").first()}")

2026-06-17 10:56:17,597 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-17 10:56:17,599 INFO sqlalchemy.engine.Engine SELECT users.id AS users_id, users.name AS users_name, users.email AS users_email, users.age AS users_age, users.created_at AS users_created_at 
FROM users 
WHERE users.name = ?
 LIMIT ? OFFSET ?
2026-06-17 10:56:17,600 INFO sqlalchemy.engine.Engine [cached since 552.5s ago] ('Alice', 1, 0)
2026-06-17 10:56:17,604 INFO sqlalchemy.engine.Engine UPDATE users SET email=? WHERE users.id = ?
2026-06-17 10:56:17,605 INFO sqlalchemy.engine.Engine [generated in 0.00057s] ('alice.new@example.com', 3)
2026-06-17 10:56:17,607 INFO sqlalchemy.engine.Engine COMMIT
2026-06-17 10:56:17,608 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-17 10:56:17,609 INFO sqlalchemy.engine.Engine SELECT users.id AS users_id, users.name AS users_name, users.email AS users_email, users.age AS users_age, users.created_at AS users_created_at 
FROM users 
WHERE users.name = ?
 LIMIT ? OFFSE

In [38]:
from sqlalchemy import update

with Session() as session:
  user = session.get(User, 1)
  if user:
    user.age = 40
    session.commit()
  
  # 일괄 업데이트
  stmt = update(User).where(User.age <= 35).values(age=20)
  session.execute(stmt)
  session.commit()

2026-06-17 11:01:42,774 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-17 11:01:42,778 INFO sqlalchemy.engine.Engine SELECT users.id AS users_id, users.name AS users_name, users.email AS users_email, users.age AS users_age, users.created_at AS users_created_at 
FROM users 
WHERE users.id = ?
2026-06-17 11:01:42,780 INFO sqlalchemy.engine.Engine [cached since 1197s ago] (1,)
2026-06-17 11:01:42,782 INFO sqlalchemy.engine.Engine COMMIT
2026-06-17 11:01:42,783 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-17 11:01:42,784 INFO sqlalchemy.engine.Engine UPDATE users SET age=? WHERE users.age <= ?
2026-06-17 11:01:42,784 INFO sqlalchemy.engine.Engine [cached since 40.32s ago] (20, 35)
2026-06-17 11:01:42,785 INFO sqlalchemy.engine.Engine COMMIT


In [41]:
with Session() as session:
  bob = session.query(User).filter_by(name='Bob').first()
  session.delete(bob)
  session.commit()

2026-06-17 11:03:03,880 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-17 11:03:03,884 INFO sqlalchemy.engine.Engine SELECT users.id AS users_id, users.name AS users_name, users.email AS users_email, users.age AS users_age, users.created_at AS users_created_at 
FROM users 
WHERE users.name = ?
 LIMIT ? OFFSET ?
2026-06-17 11:03:03,885 INFO sqlalchemy.engine.Engine [cached since 958.8s ago] ('Bob', 1, 0)
2026-06-17 11:03:03,894 INFO sqlalchemy.engine.Engine DELETE FROM users WHERE users.id = ?
2026-06-17 11:03:03,895 INFO sqlalchemy.engine.Engine [generated in 0.00183s] (1,)
2026-06-17 11:03:03,897 INFO sqlalchemy.engine.Engine COMMIT


In [42]:
from sqlalchemy import delete

# delete from 테이블명 where id=1
with Session() as session:
  stmt = delete(User).where(User.id == 4)
  session.execute(stmt)
  session.commit()

2026-06-17 11:05:05,541 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-17 11:05:05,545 INFO sqlalchemy.engine.Engine DELETE FROM users WHERE users.id = ?
2026-06-17 11:05:05,548 INFO sqlalchemy.engine.Engine [generated in 0.00081s] (4,)
2026-06-17 11:05:05,550 INFO sqlalchemy.engine.Engine COMMIT


In [ ]:
# 관계(외래키)
# 1:N, N:1, M:N
# 게시글 하나: 댓글 여러 개

from sqlalchemy import ForeignKey
from sqlalchemy.orm import relationship

engine = create_engine("sqlite:///db/users.db", echo=True)

class Base(DeclarativeBase):
    pass

class Post(Base):
  __tablename__ = "posts"
  
  id:Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
  title:Mapped[str] = mapped_column(String(200))
  content:Mapped[str]
  # 역참조
  comments:Mapped[list['Comment']] = relationship("Comment", back_populates="post", cascade='all, delete-orphan')

class Comment(Base):
  __tablename__ = "comments"
  
  id:Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
  content:Mapped[str]
  # 어느 게시글의 댓글인가
  post_id:Mapped[int] = mapped_column(ForeignKey('posts.id'))
  # 역참조
  post:Mapped['Post'] = relationship("Post", back_populates="comments") 

  

In [86]:
#테이블반영
Base.metadata.create_all(engine)

2026-06-17 12:51:32,165 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-17 12:51:32,167 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("posts")
2026-06-17 12:51:32,168 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-06-17 12:51:32,169 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("comments")
2026-06-17 12:51:32,170 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-06-17 12:51:32,172 INFO sqlalchemy.engine.Engine COMMIT


In [87]:
# 데이터 넣기
with Session() as session:
  post = Post(title="LLM입문", content="LLM이란 무엇인가?")
  post.comments = [
    Comment(content="댓글달아주세요")
  ]
  session.add(post)
  session.commit()
  
  # 역참조 이용
  p = session.get(Post, 1)
  for c in p.comments:
    print(c.content)

IndexError: list index out of range

In [ ]:
# 조회
with Session() as session:
  # 역참조 이용
  p = session.get(Post, 1)
  for c in p.comments:
    print(c.content)

In [69]:
from sqlalchemy import inspect

print(engine.url)
print(inspect(engine).get_table_names())

sqlite:///db/users.db
2026-06-17 12:33:44,733 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-06-17 12:33:44,734 INFO sqlalchemy.engine.Engine SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite~_%' ESCAPE '~' ORDER BY name
2026-06-17 12:33:44,735 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-06-17 12:33:44,736 INFO sqlalchemy.engine.Engine ROLLBACK
['comments', 'posts', 'users']


In [70]:
print(Post.__table__.columns.keys())

AttributeError: type object 'Post' has no attribute '__table__'

In [83]:
globals().pop("Post", None)
globals().pop("Comment", None)

In [84]:
Base.registry.dispose()
Base.metadata.clear()